# South Korea — Data Preprocessing

## Overview

This notebook documents the inspection, cleaning, and consolidation of South Korean bilateral trade data for the interdisciplinary project *Building a Forecasting Pipeline for Bilateral Trade Flows*.

The dataset consists of eight annual CSV files covering January 2019 to May 2026, obtained from the Observatory of Economic Complexity (OEC).

## Objectives

* Examine the structure and consistency of the source files.
* Assess temporal coverage and data quality.
* Identify and handle duplicate observations.
* Consolidate the annual files into a unified monthly dataset.
* Validate the resulting dataset and preserve reproducibility.

## Processing Workflow

1. Data inspection and schema verification
2. Temporal and missing-value assessment
3. Duplicate investigation
4. Data cleaning and aggregation
5. Export to Parquet format
6. Post-processing validation



## 1. Imports and Project Configuration


In [ ]:
import csv
from pathlib import Path

import duckdb
import pandas as pd

In [ ]:

# Project directories
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "Data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "processed"

# Korean annual files
KOREA_PATTERN = "trade_n_kor_m_hs_*.csv"
korea_files = sorted(RAW_DIR.glob(KOREA_PATTERN))

# Create processed directory if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Display paths and discovered files
print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

print("\nNumber of Korean files:", len(korea_files))

for file in korea_files:
    print(f"{file.name}: {file.stat().st_size / (1024**3):.2f} GB")

Project directory: c:\Transfer\InterdisciplinaryProject
Raw data directory: c:\Transfer\InterdisciplinaryProject\data\raw
Processed data directory: c:\Transfer\InterdisciplinaryProject\data\processed

Number of Korean files: 8
trade_n_kor_m_hs_2019.csv: 0.21 GB
trade_n_kor_m_hs_2020.csv: 0.20 GB
trade_n_kor_m_hs_2021.csv: 0.21 GB
trade_n_kor_m_hs_2022.csv: 0.21 GB
trade_n_kor_m_hs_2023.csv: 0.21 GB
trade_n_kor_m_hs_2024.csv: 0.21 GB
trade_n_kor_m_hs_2025.csv: 0.22 GB
trade_n_kor_m_hs_2026.csv: 0.09 GB


## 2. Schema Consistency Across Annual Files
This section check whether the same schema remains accross all korea dataset files.

In [21]:
reference_columns = None
schema_consistency = True

for file in korea_files:
    with open(file, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        columns = next(reader)

    if reference_columns is None:
        reference_columns = columns
        print(f"Reference schema: {file.name}")
        print(f"Columns: {reference_columns}")
    else:
        if columns == reference_columns:
            print(f"{file.name}: Schema consistent")
        else:
            schema_consistency = False
            print(f"{file.name}: Schema mismatch")
            print(f"Missing columns: {set(reference_columns) - set(columns)}")
            print(f"Additional columns: {set(columns) - set(reference_columns)}")
            print(f"Column order differs: {columns != reference_columns}")

print(f"\nOverall schema consistency: {schema_consistency}")

Reference schema: trade_n_kor_m_hs_2019.csv
Columns: ['month_id', 'product_id', 'product_name', 'country_id', 'country_name', 'trade_flow_name', 'trade_value', 'weight']
trade_n_kor_m_hs_2020.csv: Schema consistent
trade_n_kor_m_hs_2021.csv: Schema consistent
trade_n_kor_m_hs_2022.csv: Schema consistent
trade_n_kor_m_hs_2023.csv: Schema consistent
trade_n_kor_m_hs_2024.csv: Schema consistent
trade_n_kor_m_hs_2025.csv: Schema consistent
trade_n_kor_m_hs_2026.csv: Schema consistent

Overall schema consistency: True


## 3. Data integrity: Temporal consistency, malformed rows, duplicate-key detection.


### 3.1 Temporal Consistency Check
This section examines the temporal coverage of the source files to verify that the monthly observations follow the expected reporting periods without gaps.

In [22]:
def check_month_continuity(path):
    months = set()

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        month_idx = header.index("month_id")

        for row in reader:
            month = int(row[month_idx])
            months.add(month)

    year = int(path.stem.split("_")[-1])

    expected_months = {
        year * 100 + month
        for month in range(1, 13)
    }

    # 2026 is only expected to contain January–May
    if year == 2026:
        expected_months = {
            year * 100 + month
            for month in range(1, 6)
        }

    missing_months = expected_months - months
    unexpected_months = months - expected_months

    return {
        "file": path.name,
        "observed_months": len(months),
        "missing_months": sorted(missing_months),
        "unexpected_months": sorted(unexpected_months),
        "continuous": months == expected_months
    }


for file in korea_files:
    result = check_month_continuity(file)

    print(
        f"{result['file']}: "
        f"{result['observed_months']} months | "
        f"Missing: {result['missing_months']} | "
        f"Unexpected: {result['unexpected_months']} | "
        f"Continuous: {result['continuous']}"
    )

trade_n_kor_m_hs_2019.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2020.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2021.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2022.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2023.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2024.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2025.csv: 12 months | Missing: [] | Unexpected: [] | Continuous: True
trade_n_kor_m_hs_2026.csv: 5 months | Missing: [] | Unexpected: [] | Continuous: True


### 3.2 Malformed Row Check
This section checks the source data for malformed records and verifies that the CSV structure is consistent with the expected schema.

In [23]:
def check_file_structure(path):
    expected_columns = [
        "month_id",
        "product_id",
        "product_name",
        "country_id",
        "country_name",
        "trade_flow_name",
        "trade_value",
        "weight"
    ]

    total_rows = 0
    malformed_rows = 0
    first_bad_row = None

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)

        header = next(reader)

        # Check header
        header_ok = header == expected_columns

        for line_number, row in enumerate(reader, start=2):
            total_rows += 1

            if len(row) != len(expected_columns):
                malformed_rows += 1

                if first_bad_row is None:
                    first_bad_row = (line_number, len(row), row)

    return {
        "header_ok": header_ok,
        "total_rows": total_rows,
        "malformed_rows": malformed_rows,
        "first_bad_row": first_bad_row
    }

In [24]:
for file in korea_files:
    result = check_file_structure(file)

    print(f"\n{file.name}")
    print(f"  Header correct: {result['header_ok']}")
    print(f"  Total data rows: {result['total_rows']:,}")
    print(f"  Malformed rows: {result['malformed_rows']:,}")

    if result["first_bad_row"] is not None:
        print(f"  First malformed row: {result['first_bad_row']}")


trade_n_kor_m_hs_2019.csv
  Header correct: True
  Total data rows: 2,265,154
  Malformed rows: 0

trade_n_kor_m_hs_2020.csv
  Header correct: True
  Total data rows: 2,214,324
  Malformed rows: 0

trade_n_kor_m_hs_2021.csv
  Header correct: True
  Total data rows: 2,299,326
  Malformed rows: 0

trade_n_kor_m_hs_2022.csv
  Header correct: True
  Total data rows: 2,286,302
  Malformed rows: 0

trade_n_kor_m_hs_2023.csv
  Header correct: True
  Total data rows: 2,305,464
  Malformed rows: 0

trade_n_kor_m_hs_2024.csv
  Header correct: True
  Total data rows: 2,361,846
  Malformed rows: 0

trade_n_kor_m_hs_2025.csv
  Header correct: True
  Total data rows: 2,396,044
  Malformed rows: 0

trade_n_kor_m_hs_2026.csv
  Header correct: True
  Total data rows: 988,652
  Malformed rows: 0


### 3.3 Initial data completeness

The completeness of the core analytical variables is assessed by counting missing values in the monthly identifier, product identifier, trading-partner identifier, trade-flow variable, and trade value.

In [25]:
con = duckdb.connect(database=":memory:")
null_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE month_id IS NULL) AS missing_month_id,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_id,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_id,
        COUNT(*) FILTER (WHERE trade_flow_name IS NULL) AS missing_trade_flow,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_value,
        COUNT(*) FILTER (WHERE weight IS NULL) AS missing_weight
    FROM read_csv_auto(?, union_by_name = TRUE)
""", [[str(file) for file in korea_files]]).df()

display(null_check.T)

,0
total_rows,17117112
missing_month_id,0
missing_product_id,0
missing_country_id,0
missing_trade_flow,0
missing_trade_value,0
missing_weight,0


### 3.4 Duplicate-Key Detection
This section identifies records sharing the same analytical key to determine the extent of potential duplication in the dataset.

In [26]:
def check_duplicate_keys(path):
    seen = set()
    duplicate_count = 0
    first_duplicate = None

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for line_number, row in enumerate(reader, start=2):
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            if key in seen:
                duplicate_count += 1

                if first_duplicate is None:
                    first_duplicate = (line_number, key)

            else:
                seen.add(key)

    return duplicate_count, first_duplicate

In [27]:
for file in korea_files:
    duplicates, first_duplicate = check_duplicate_keys(file)

    print(f"\n{file.name}")
    print(f"  Duplicate keys: {duplicates:,}")

    if first_duplicate is not None:
        print(f"  First duplicate: {first_duplicate}")


trade_n_kor_m_hs_2019.csv
  Duplicate keys: 1,142
  First duplicate: (1014, ('201901', '999999', 'che', 'Imports'))

trade_n_kor_m_hs_2020.csv
  Duplicate keys: 1,068
  First duplicate: (3090, ('202001', '999999', 'jpn', 'Imports'))

trade_n_kor_m_hs_2021.csv
  Duplicate keys: 1,096
  First duplicate: (3847, ('202101', '999999', 'tha', 'Imports'))

trade_n_kor_m_hs_2022.csv
  Duplicate keys: 2,274
  First duplicate: (1035, ('202201', '852491', 'egy', 'Imports'))

trade_n_kor_m_hs_2023.csv
  Duplicate keys: 1,052
  First duplicate: (2475, ('202301', '999999', 'moz', 'Imports'))

trade_n_kor_m_hs_2024.csv
  Duplicate keys: 88
  First duplicate: (4847, ('202401', '999999', 'zaf', 'Imports'))

trade_n_kor_m_hs_2025.csv
  Duplicate keys: 10
  First duplicate: (252820, ('202502', '870323', 'maf', 'Imports'))

trade_n_kor_m_hs_2026.csv
  Duplicate keys: 10
  First duplicate: (66214, ('202601', '300490', 'maf', 'Imports'))


## 4. Duplicate investigation: Duplicate classification, conflicting records, trade-value and weight patterns.

### 4.1 Inspection of Duplicate Records
This section examines duplicated records in detail to understand their structure and determine whether they represent identical or distinct observations.

In [28]:
def find_duplicate_rows(path, target_key):
    matches = []

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            if key == target_key:
                matches.append(row)

    return matches

In [29]:
target_key = ('201901', '999999', 'che', 'Imports')

duplicates = find_duplicate_rows(
    korea_files[0],
    target_key
)

for row in duplicates:
    print(row)

{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'che', 'country_name': 'Switzerland', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '2065'}
{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'che', 'country_name': 'Switzerland', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '0'}


In [30]:
target_key = ('202201', '852491', 'egy', 'Imports')

duplicates = find_duplicate_rows(
    RAW_DIR / "trade_n_kor_m_hs_2022.csv",
    target_key
)

for row in duplicates:
    print(row)

{'month_id': '202201', 'product_id': '852491', 'product_name': 'Media, recorded; n.e.s. in heading no. 8524, for reproducing phenomena other than sound or image, (excluding products of chapter 37)', 'country_id': 'egy', 'country_name': 'Egypt', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '2'}
{'month_id': '202201', 'product_id': '852491', 'product_name': 'Media, recorded; n.e.s. in heading no. 8524, for reproducing phenomena other than sound or image, (excluding products of chapter 37)', 'country_id': 'egy', 'country_name': 'Egypt', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '2'}


### 4.2 Classification of Duplicate Records
This section classifies duplicate-key groups according to whether their records contain identical or differing trade values and weights.

In [31]:
def analyze_duplicates(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            row_tuple = tuple(row.items())

            if key not in groups:
                groups[key] = []

            groups[key].append(row_tuple)

    exact_duplicates = 0
    conflicting_duplicates = 0

    for key, rows in groups.items():
        if len(rows) > 1:
            unique_rows = set(rows)

            if len(unique_rows) == 1:
                exact_duplicates += len(rows) - 1
            else:
                conflicting_duplicates += len(rows) - 1

    return exact_duplicates, conflicting_duplicates

In [32]:
for file in korea_files:
    exact, conflicting = analyze_duplicates(file)

    print(f"\n{file.name}")
    print(f"  Exact duplicate rows: {exact:,}")
    print(f"  Conflicting duplicate rows: {conflicting:,}")


trade_n_kor_m_hs_2019.csv
  Exact duplicate rows: 0
  Conflicting duplicate rows: 1,142

trade_n_kor_m_hs_2020.csv
  Exact duplicate rows: 0
  Conflicting duplicate rows: 1,068

trade_n_kor_m_hs_2021.csv
  Exact duplicate rows: 0
  Conflicting duplicate rows: 1,096

trade_n_kor_m_hs_2022.csv
  Exact duplicate rows: 1,181
  Conflicting duplicate rows: 1,093

trade_n_kor_m_hs_2023.csv
  Exact duplicate rows: 2
  Conflicting duplicate rows: 1,050

trade_n_kor_m_hs_2024.csv
  Exact duplicate rows: 6
  Conflicting duplicate rows: 82

trade_n_kor_m_hs_2025.csv
  Exact duplicate rows: 5
  Conflicting duplicate rows: 5

trade_n_kor_m_hs_2026.csv
  Exact duplicate rows: 5
  Conflicting duplicate rows: 5


### 4.3 Identification of Conflicting Duplicate Records
This section identifies duplicate-key groups containing inconsistent values that require further investigation before consolidation.

In [33]:
def find_conflicting_duplicate(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            row_tuple = tuple(row.items())

            if key not in groups:
                groups[key] = []

            groups[key].append(row_tuple)

    for key, rows in groups.items():
        if len(rows) > 1 and len(set(rows)) > 1:
            return key, rows

    return None, None

In [34]:
key, rows = find_conflicting_duplicate(korea_files[0])

print("Conflicting key:")
print(key)

print("\nRows:")
for row in rows:
    print(dict(row))

Conflicting key:
('201901', '999999', 'che', 'Imports')

Rows:
{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'che', 'country_name': 'Switzerland', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '2065'}
{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'che', 'country_name': 'Switzerland', 'trade_flow_name': 'Imports', 'trade_value': '0', 'weight': '0'}


### 4.4 Investigation of Trade-Value Differences Across Duplicate Records
This section examines the magnitude and distribution of trade-value differences within conflicting duplicate groups.

In [35]:
def analyze_duplicate_trade_values(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            if key not in groups:
                groups[key] = []

            groups[key].append(row)

    different_trade_values = []

    for key, rows in groups.items():
        if len(rows) > 1:
            trade_values = {row["trade_value"] for row in rows}

            if len(trade_values) > 1:
                different_trade_values.append(
                    (key, rows)
                )

    return different_trade_values

In [36]:
different_values = analyze_duplicate_trade_values(korea_files[0])

print("Duplicate groups with different trade values:",
      len(different_values))

if different_values:
    print("\nFirst example:")
    key, rows = different_values[0]

    print("Key:", key)

    for row in rows:
        print(row)

Duplicate groups with different trade values: 553

First example:
Key: ('201901', '999999', 'mus', 'Exports')
{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'mus', 'country_name': 'Mauritius', 'trade_flow_name': 'Exports', 'trade_value': '1000', 'weight': '1350'}
{'month_id': '201901', 'product_id': '999999', 'product_name': 'Commodities not specified according to kind', 'country_id': 'mus', 'country_name': 'Mauritius', 'trade_flow_name': 'Exports', 'trade_value': '0', 'weight': '0'}


### 4.5 Comparison of Conflicting Records Across All Fields
This section compares conflicting records across their available attributes to determine which fields differ within the same analytical key.

In [37]:
def summarize_duplicate_differences(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            groups.setdefault(key, []).append(row)

    difference_patterns = {}

    for key, rows in groups.items():
        if len(rows) > 1:
            # Compare all fields other than the proposed key
            differences = []

            for field in [
                "product_name",
                "country_name",
                "trade_value",
                "weight"
            ]:
                values = {row[field] for row in rows}

                if len(values) > 1:
                    differences.append(field)

            pattern = tuple(differences)
            difference_patterns[pattern] = difference_patterns.get(pattern, 0) + 1

    return difference_patterns

In [38]:
patterns = summarize_duplicate_differences(korea_files[0])

print("2019 duplicate difference patterns:\n")

for pattern, count in sorted(
    patterns.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{pattern}: {count:,} groups")

2019 duplicate difference patterns:

('weight',): 589 groups
('trade_value', 'weight'): 553 groups


## 5. Value and weight validation: Zero values, positive trade values, and weight consistency.

### 5.1 Trade-Value and Weight Patterns in Duplicate Records
This section investigates the relationship between trade values and reported weights across duplicated observations to identify recurring data patterns.

In [39]:
def analyze_duplicate_zero_patterns(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            groups.setdefault(key, []).append(row)

    patterns = {}

    for key, rows in groups.items():
        if len(rows) > 1:
            pattern = tuple(
                (
                    row["trade_value"] != "0",
                    row["weight"] != "0"
                )
                for row in rows
            )

            patterns[pattern] = patterns.get(pattern, 0) + 1

    return patterns

In [40]:
patterns = analyze_duplicate_zero_patterns(korea_files[0])

print("2019 duplicate value/weight patterns:\n")

for pattern, count in sorted(
    patterns.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{pattern}: {count:,} groups")

2019 duplicate value/weight patterns:

((False, False), (False, True)): 556 groups
((True, True), (False, False)): 536 groups
((False, True), (False, False)): 33 groups
((False, False), (True, True)): 17 groups



### 5.2 Frequency of Zero-Value Records Across the Dataset
This section measures the frequency of zero trade-value observations and examines their distribution across the dataset.

In [41]:
def count_zero_values(path):
    total = 0
    zero_trade = 0
    zero_weight = 0
    both_zero = 0

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            total += 1

            trade_zero = row["trade_value"] == "0"
            weight_zero = row["weight"] == "0"

            if trade_zero:
                zero_trade += 1

            if weight_zero:
                zero_weight += 1

            if trade_zero and weight_zero:
                both_zero += 1

    return total, zero_trade, zero_weight, both_zero

In [42]:
total, zero_trade, zero_weight, both_zero = count_zero_values(korea_files[0])

print(f"Total rows: {total:,}")
print(f"Trade value = 0: {zero_trade:,}")
print(f"Weight = 0: {zero_weight:,}")
print(f"Both = 0: {both_zero:,}")

Total rows: 2,265,154
Trade value = 0: 1,076,093
Weight = 0: 883,173
Both = 0: 874,058


### 5.3 Validation of Positive Trade Values Against Weight
This section examines whether records with positive trade values are accompanied by valid positive weight measurements.

In [43]:
def analyze_trade_weight_relationship(path):
    total = 0
    positive_trade = 0
    positive_trade_zero_weight = 0
    zero_trade_positive_weight = 0

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            total += 1

            trade_positive = int(row["trade_value"]) > 0
            weight_positive = int(row["weight"]) > 0

            if trade_positive:
                positive_trade += 1

                if not weight_positive:
                    positive_trade_zero_weight += 1

            if not trade_positive and weight_positive:
                zero_trade_positive_weight += 1

    return (
        total,
        positive_trade,
        positive_trade_zero_weight,
        zero_trade_positive_weight
    )

In [44]:
total, positive_trade, positive_trade_zero_weight, zero_trade_positive_weight = (
    analyze_trade_weight_relationship(korea_files[0])
)

print(f"Total rows: {total:,}")
print(f"Positive trade value: {positive_trade:,}")
print(f"Positive trade + zero weight: {positive_trade_zero_weight:,}")
print(f"Zero trade value + positive weight: {zero_trade_positive_weight:,}")

Total rows: 2,265,154
Positive trade value: 1,189,061
Positive trade + zero weight: 9,115
Zero trade value + positive weight: 202,035



### 5.4 Investigation of Multiple Positive Trade Values
This section investigates duplicate-key groups containing multiple positive trade values to understand how these observations contribute to the overall trade totals.

In [45]:
def find_distinct_multiple_positive_trade_values(path):
    groups = {}

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (
                row["month_id"],
                row["product_id"],
                row["country_id"],
                row["trade_flow_name"]
            )

            groups.setdefault(key, []).append(row)

    results = []

    for key, rows in groups.items():
        positive_rows = [
            row for row in rows
            if int(row["trade_value"]) > 0
        ]

        # Remove completely identical rows
        unique_positive_rows = {
            tuple(row.items())
            for row in positive_rows
        }

        if len(unique_positive_rows) > 1:
            results.append((key, positive_rows))

    return results

In [46]:
for file in korea_files:
    results = find_distinct_multiple_positive_trade_values(file)

    print(
        f"{file.name}: "
        f"{len(results):,} groups with multiple DISTINCT positive trade-value rows"
    )

trade_n_kor_m_hs_2019.csv: 0 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2020.csv: 0 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2021.csv: 0 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2022.csv: 0 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2023.csv: 1 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2024.csv: 5 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2025.csv: 5 groups with multiple DISTINCT positive trade-value rows
trade_n_kor_m_hs_2026.csv: 4 groups with multiple DISTINCT positive trade-value rows


### 5.5 Inspection of Multiple Trade-Value Groups
This section examines groups containing multiple distinct trade values to characterize their underlying record structure and potential aggregation behavior.

In [47]:
for file in korea_files:
    results = find_distinct_multiple_positive_trade_values(file)

    if results:
        print(f"\n{file.name}")
        print(f"Number of groups: {len(results)}")

        for key, rows in results:
            print("\nKey:", key)

            for row in rows:
                print(
                    f"  trade_value={row['trade_value']}, "
                    f"weight={row['weight']}, "
                    f"product={row['product_name']}"
                )


trade_n_kor_m_hs_2023.csv
Number of groups: 1

Key: ('202301', '330499', 'maf', 'Exports')
  trade_value=3000, weight=7, product=Beauty, makeup and suntan preparations nes
  trade_value=5000, weight=10, product=Beauty, makeup and suntan preparations nes

trade_n_kor_m_hs_2024.csv
Number of groups: 5

Key: ('202403', '870323', 'maf', 'Exports')
  trade_value=492000, weight=43251, product=Medium Sized Cars
  trade_value=11000, weight=1392, product=Medium Sized Cars

Key: ('202404', '870323', 'maf', 'Exports')
  trade_value=382000, weight=34330, product=Medium Sized Cars
  trade_value=9000, weight=1460, product=Medium Sized Cars

Key: ('202408', '870421', 'maf', 'Exports')
  trade_value=349000, weight=39555, product=Diesel Trucks < 5 tonnes
  trade_value=5000, weight=2985, product=Diesel Trucks < 5 tonnes

Key: ('202411', '870323', 'maf', 'Exports')
  trade_value=9000, weight=1300, product=Medium Sized Cars
  trade_value=1473000, weight=119276, product=Medium Sized Cars

Key: ('202411', 


### 5.6 Exact Duplicate Count

In [48]:
def count_exact_duplicates(path):
    seen = set()
    exact_duplicates = 0

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            row_tuple = tuple(row)

            if row_tuple in seen:
                exact_duplicates += 1
            else:
                seen.add(row_tuple)

    return exact_duplicates

In [49]:
for file in korea_files:
    duplicates = count_exact_duplicates(file)

    print(
        f"{file.name}: "
        f"{duplicates:,} exact duplicate rows"
    )

trade_n_kor_m_hs_2019.csv: 0 exact duplicate rows
trade_n_kor_m_hs_2020.csv: 0 exact duplicate rows
trade_n_kor_m_hs_2021.csv: 0 exact duplicate rows
trade_n_kor_m_hs_2022.csv: 1,181 exact duplicate rows
trade_n_kor_m_hs_2023.csv: 2 exact duplicate rows
trade_n_kor_m_hs_2024.csv: 6 exact duplicate rows
trade_n_kor_m_hs_2025.csv: 5 exact duplicate rows
trade_n_kor_m_hs_2026.csv: 5 exact duplicate rows


## 6. Consolidation of South Korea trade data using DuckDB into a unified dataset

### 6.1 Consistency Check Between the DuckDB In-Memory Database and Source CSV Files
This section compares key dataset statistics obtained through DuckDB with the original CSV files to verify the consistency of the data-reading process.

In [52]:
# Prepare a list of CSV file paths
korea_paths = [str(file) for file in korea_files]

# Read all annual files as one logical dataset
korea_query = """
    SELECT *
    FROM read_csv(
        ?,
        header = true,
        union_by_name = true,
        sample_size = 100000
    )
"""

# Count total records
total_rows = con.execute(
    """
    SELECT COUNT(*)
    FROM read_csv(?, header = true, union_by_name = true)
    """,
    [korea_paths]
).fetchone()[0]

print(f"Total records: {total_rows:,}")

Total records: 17,117,112


In [53]:
korea_summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS unique_months,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT product_id) AS unique_products,
        COUNT(DISTINCT country_id) AS unique_countries,
        COUNT(DISTINCT trade_flow_name) AS unique_trade_flows,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_value,
        COUNT(*) FILTER (WHERE weight IS NULL) AS missing_weight,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_id,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_id
    FROM read_csv(
        ?,
        header = true,
        union_by_name = true
    )
""", [korea_paths]).df()

display(korea_summary)

,total_rows,unique_months,first_month,last_month,unique_products,unique_countries,unique_trade_flows,missing_trade_value,missing_weight,missing_product_id,missing_country_id
0,17117112,89,201901,202605,5706,250,2,0,0,0,0


In [54]:
duplicate_summary = con.execute("""
    WITH source_data AS (
        SELECT *
        FROM read_csv(
            ?,
            header = true,
            union_by_name = true
        )
    ),

    exact_duplicates AS (
        SELECT
            COUNT(*) - COUNT(DISTINCT (
                month_id,
                product_id,
                product_name,
                country_id,
                country_name,
                trade_flow_name,
                trade_value,
                weight
            )) AS exact_duplicate_rows
        FROM source_data
    ),

    key_groups AS (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name,
            COUNT(*) AS records_in_group,
            COUNT(DISTINCT (
                trade_value,
                weight
            )) AS distinct_value_weight_pairs
        FROM source_data
        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
    )

    SELECT
        (SELECT exact_duplicate_rows FROM exact_duplicates)
            AS exact_duplicate_rows,

        COUNT(*) FILTER (
            WHERE records_in_group > 1
        ) AS duplicated_key_groups,

        SUM(records_in_group - 1) FILTER (
            WHERE records_in_group > 1
        ) AS excess_records_from_key_duplicates,

        COUNT(*) FILTER (
            WHERE distinct_value_weight_pairs > 1
        ) AS groups_with_distinct_values

    FROM key_groups
""", [korea_paths]).df()

display(duplicate_summary)

,exact_duplicate_rows,duplicated_key_groups,excess_records_from_key_duplicates,groups_with_distinct_values
0,1199,6740,6740.0,5541


In [55]:
description_check = con.execute("""
    WITH source_data AS (
        SELECT *
        FROM read_csv(
            ?,
            header = true,
            union_by_name = true
        )
    )

    SELECT
        COUNT(*) AS total_key_groups,

        COUNT(*) FILTER (
            WHERE product_name_variants > 1
        ) AS groups_with_multiple_product_names,

        COUNT(*) FILTER (
            WHERE country_name_variants > 1
        ) AS groups_with_multiple_country_names

    FROM (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name,
            COUNT(DISTINCT product_name) AS product_name_variants,
            COUNT(DISTINCT country_name) AS country_name_variants

        FROM source_data

        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
    )
""", [korea_paths]).df()

display(description_check)

,total_key_groups,groups_with_multiple_product_names,groups_with_multiple_country_names
0,17110372,0,0


### 6.2 Consolidation of South Korean Trade Data
This section consolidates the annual South Korean trade files into a unified dataset while applying the established duplicate-handling procedure.

In [56]:
# Output location
KOREA_OUTPUT = PROCESSED_DIR / "korea_trade_monthly.parquet"

# Consolidate, deduplicate, and aggregate
con.execute("""
    COPY (
        SELECT
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            SUM(trade_value) AS trade_value,
            SUM(weight) AS weight

        FROM (
            SELECT DISTINCT *
            FROM read_csv(
                ?,
                header = true,
                union_by_name = true
            )
        )

        GROUP BY
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name

    )
    TO '""" + str(KOREA_OUTPUT).replace("\\", "/") + """'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""", [korea_paths])

print("Consolidated dataset saved successfully.")
print("Output:", KOREA_OUTPUT)
print(f"File size: {KOREA_OUTPUT.stat().st_size / (1024**2):.2f} MB")

Consolidated dataset saved successfully.
Output: c:\Transfer\InterdisciplinaryProject\data\processed\korea_trade_monthly.parquet
File size: 156.09 MB



### 6.3 Validation of the Processed Parquet Dataset Against the Original CSV Findings
This section validates the processed Parquet dataset against the source data by comparing its structure, temporal coverage, duplicate status, and aggregate trade measures.

In [60]:
# Inspect the processed Parquet dataset
# Inspect the processed Parquet dataset
OUTPUT_FILE = PROCESSED_DIR / "korea_trade_monthly.parquet"

validation = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS unique_months,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT product_id) AS unique_products,
        COUNT(DISTINCT country_id) AS unique_countries,
        COUNT(*) FILTER (WHERE month_id IS NULL) AS missing_month_id,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_id,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_id,
        COUNT(*) FILTER (WHERE trade_flow_name IS NULL) AS missing_trade_flow,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_value,
        COUNT(*) FILTER (WHERE weight IS NULL) AS missing_weight,
        SUM(trade_value) AS total_trade_value,
        SUM(weight) AS total_weight
    FROM read_parquet(?)
""", [str(OUTPUT_FILE)]).df()

display(validation.T)

,0
total_rows,1.711037e+07
unique_months,8.900000e+01
first_month,2.019010e+05
last_month,2.026050e+05
unique_products,5.706000e+03
unique_countries,2.500000e+02
missing_month_id,0.000000e+00
missing_product_id,0.000000e+00
missing_country_id,0.000000e+00
missing_trade_flow,0.000000e+00


In [58]:
duplicate_check = con.execute("""
    SELECT COUNT(*) AS duplicated_key_groups
    FROM (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name
        FROM read_parquet(?)
        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
        HAVING COUNT(*) > 1
    )
""", [str(KOREA_OUTPUT)]).fetchone()[0]

print("Duplicated analytical key groups:", duplicate_check)

Duplicated analytical key groups: 0


In [59]:
source_totals = con.execute("""
    SELECT
        SUM(trade_value) AS source_trade_value,
        SUM(weight) AS source_weight
    FROM (
        SELECT DISTINCT *
        FROM read_csv(
            ?,
            header = true,
            union_by_name = true
        )
    )
""", [korea_paths]).fetchone()

processed_totals = con.execute("""
    SELECT
        SUM(trade_value),
        SUM(weight)
    FROM read_parquet(?)
""", [str(KOREA_OUTPUT)]).fetchone()

print("SOURCE TOTALS (after exact deduplication)")
print("Trade value:", source_totals[0])
print("Weight:", source_totals[1])

print("\nPROCESSED TOTALS")
print("Trade value:", processed_totals[0])
print("Weight:", processed_totals[1])

print("\nVALIDATION")
print("Trade value preserved:", source_totals[0] == processed_totals[0])
print("Weight preserved:", source_totals[1] == processed_totals[1])

SOURCE TOTALS (after exact deduplication)
Trade value: 9308798185000
Weight: 5577999599483

PROCESSED TOTALS
Trade value: 9308798185000.0
Weight: 5577999599483.0

VALIDATION
Trade value preserved: True
Weight preserved: True


# Data Preparation and Consolidation — South Korea

## 1. Objective

The objective of this stage was to inspect, clean, and consolidate the annual South Korean bilateral trade datasets into a unified monthly dataset suitable for subsequent exploratory data analysis and forecasting.

The processing workflow focused on:

* Verifying consistency across annual source files.
* Examining temporal coverage and data completeness.
* Identifying and investigating duplicate observations.
* Removing exact duplicate records.
* Aggregating observations according to the intended analytical key.
* Preserving the original trade values and reported weights.
* Exporting the consolidated dataset in Parquet format.
* Validating the resulting dataset.

The original source files were retained without modification.

## 2. Data Source and Input Files

The South Korean trade data were obtained from the Observatory of Economic Complexity (OEC) dataset collection.

Eight annual CSV files were used, covering 2019 through May 2026.

| File                        | Temporal coverage     |
| --------------------------- | --------------------- |
| `trade_n_kor_m_hs_2019.csv` | January–December 2019 |
| `trade_n_kor_m_hs_2020.csv` | January–December 2020 |
| `trade_n_kor_m_hs_2021.csv` | January–December 2021 |
| `trade_n_kor_m_hs_2022.csv` | January–December 2022 |
| `trade_n_kor_m_hs_2023.csv` | January–December 2023 |
| `trade_n_kor_m_hs_2024.csv` | January–December 2024 |
| `trade_n_kor_m_hs_2025.csv` | January–December 2025 |
| `trade_n_kor_m_hs_2026.csv` | January–May 2026      |

The combined raw data volume was approximately 1.57 GB.

### Original schema

All eight files shared the same column structure:

| Variable          | Description                            |
| ----------------- | -------------------------------------- |
| `month_id`        | Monthly observation identifier         |
| `product_id`      | Product classification code            |
| `product_name`    | Product description                    |
| `country_id`      | Trading partner identifier             |
| `country_name`    | Trading partner name                   |
| `trade_flow_name` | Trade flow, such as imports or exports |
| `trade_value`     | Reported trade value                   |
| `weight`          | Reported trade weight                  |

The product identifiers were observed at six-digit classification level.

## 3. Data Inspection and Quality Assessment

The eight annual files were inspected using Python and DuckDB.

DuckDB was selected for the data-processing operations because it enables analytical queries over large CSV files without requiring the entire dataset to be loaded into pandas memory.

### 3.1 Schema consistency

The headers and column ordering of all eight annual files were compared.

**Result:** All files shared an identical schema, allowing them to be combined into one logical dataset.

### 3.2 Temporal coverage

The combined dataset was inspected for its minimum and maximum observation periods and number of distinct months.

**Results:**

* First observation: January 2019 (`201901`)
* Last observation: May 2026 (`202605`)
* Number of distinct months: 89

The annual files exhibited the expected monthly coverage, with no visible gaps or overlaps in the previously conducted temporal checks.

### 3.3 Initial data completeness


The following analytical variables were examined for missing values across all eight annual source files:

month_id

product_id

country_id

trade_flow_name

trade_value

weight

No missing values were detected in these variables across the 17,117,112 source observations.

### 3.4 Initial dataset statistics

| Metric                    |     Result |
| ------------------------- | ---------: |
| Original observations     | 17,117,112 |
| Distinct months           |         89 |
| Distinct products         |      5,706 |
| Distinct trading partners |        250 |
| Distinct trade flows      |          2 |
| Missing trade values      |          0 |
| Missing weights           |          0 |
| Missing product IDs       |          0 |
| Missing country IDs       |          0 |

## 4. Duplicate Investigation

Duplicate observations were investigated using the following proposed analytical key:

`month_id + product_id + country_id + trade_flow_name`

The purpose was to determine whether multiple records represented exact repetitions or distinct observations that needed to be consolidated.

### 4.1 Exact duplicate records

An exact duplicate was defined as a record identical across all eight original columns.

The investigation identified:

**1,199 exact duplicate rows.**

These records were removed during consolidation to avoid double counting.

### 4.2 Duplicate analytical keys

The duplicate investigation identified:

* 6,740 duplicated analytical key groups.
* 6,740 excess records associated with duplicated keys.
* 5,541 groups containing distinct trade-value and weight combinations.

The results demonstrated that not all duplicated analytical keys represented erroneous records.

Some records contained distinct reported values and therefore could not be removed solely on the basis of the analytical key.

### 4.3 Descriptive consistency

The consistency of `product_name` and `country_name` was checked within each analytical key.

| Check                              | Result |
| ---------------------------------- | -----: |
| Groups with multiple product names |      0 |
| Groups with multiple country names |      0 |

No conflicting product or country descriptions were identified within the examined groups.

## 5. Data Cleaning and Consolidation

The processing procedure consisted of the following operations.

### Step 1: Combine annual files

The eight annual CSV files were read together using DuckDB.

### Step 2: Remove exact duplicates

Exact duplicate rows were removed using `SELECT DISTINCT` across all original columns.

### Step 3: Aggregate observations

The deduplicated records were grouped by:

* `month_id`
* `product_id`
* `product_name`
* `country_id`
* `country_name`
* `trade_flow_name`

The numerical variables were aggregated using summation:

$$
TradeValue_{aggregated} = \sum_i TradeValue_i
$$

$$
Weight_{aggregated} = \sum_i Weight_i
$$

This produced one record per analytical key.

### Step 4: Export the consolidated dataset

The resulting dataset was exported in Parquet format using Zstandard compression.

The output was stored at:

`data/processed/korea_trade_monthly.parquet`

The original CSV files in `data/raw/` were retained unchanged.

## 6. Post-Processing Validation

The resulting Parquet dataset was subjected to structural and numerical validation.

### 6.1 Structural validation

| Metric                          |     Result |
| ------------------------------- | ---------: |
| Processed observations          | 17,110,372 |
| Distinct months                 |         89 |
| First month                     |     201901 |
| Last month                      |     202605 |
| Distinct products               |      5,706 |
| Distinct trading partners       |        250 |
| Missing trade values            |          0 |
| Missing weights                 |          0 |
| Duplicate analytical key groups |          0 |

### 6.2 Aggregate preservation

The total trade value and reported weight were compared between:

1. The source records after exact duplicate removal.
2. The consolidated Parquet dataset.

| Measure           | Source after deduplication | Processed dataset |
| ----------------- | -------------------------: | ----------------: |
| Total trade value |          9,308,798,185,000 | 9,308,798,185,000 |
| Total weight      |          5,577,999,599,483 | 5,577,999,599,483 |

Both aggregate preservation checks returned `True`.

This confirms that the consolidation procedure preserved the total reported trade value and weight after exact duplicate removal.

### 6.3 Storage efficiency

The consolidated dataset occupies approximately 156.06 MB.

The output is substantially smaller than the approximately 1.57 GB of original CSV files due to the use of the Parquet columnar format and Zstandard compression.

## 7. Final Output

The resulting dataset contains monthly South Korean trade observations organized by product, trading partner, and trade flow.

**Output file:** `korea_trade_monthly.parquet`

**Format:** Parquet

**Compression:** Zstandard

**Number of observations:** 17,110,372

**Temporal coverage:** January 2019 – May 2026

The dataset has passed the structural, uniqueness, missing-value, and aggregate preservation checks conducted during this preparation stage.

## 8. Remaining Methodological Considerations

Although the consolidation and validation procedures were successful, several considerations remain relevant before the dataset is used for forecasting.

### Weight aggregation

The total reported weight was preserved during aggregation. However, the interpretation and additivity of weight across source records should be confirmed against the source documentation before using it as an analytical variable.

### Zero trade values

Zero trade values were retained. Their frequency, interpretation, and potential implications for forecasting metrics will be examined during EDA and subsequent model preparation.

### Forecasting series completeness

The dataset contains 89 distinct months overall. This does not imply that every product–partner–trade-flow combination has an observation for every month.

Missing periods within individual forecasting series will therefore require separate investigation.

## 9. Conclusion

The South Korean annual trade files were successfully consolidated into a single processed dataset.

The resulting data preserve the total reported trade value and weight after exact duplicate removal, contain no duplicated analytical keys, and exhibit the expected overall monthly coverage.

The consolidated dataset is ready for the next stage of the project: applying the corresponding data preparation procedure to the Slovak and US trade datasets, followed by cross-country consistency validation before comparative exploratory data analysis.
